# 1、流式调用(stream)

举例：

In [1]:
from langchain.chat_models import init_chat_model
import os
from dotenv import load_dotenv

# 加载配置文件
load_dotenv(override=True)

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")

# 获取大模型
model = init_chat_model(
    # model="deepseek-v4-flash",
    # model_provider="deepseek",
    # 或者
    model="deepseek:deepseek-flash",
    temperature=1.9,
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL,
    max_tokens=10000,
)

In [3]:
for chunk in model.stream("帮我解释一下什么是人工智能？"):
    print(chunk.text,end="",flush=True)

人工智能（AI）简单说，就是**让机器具备类似人类智能的能力，从而完成需要“感知、学习、推理、决策、交流”等任务**。

可以把它拆成三层理解：

1. **目标**：让机器做原本需要人脑完成的事  
   比如识别图片、听懂语音、翻译语言、下棋、写文章、推荐内容、自动驾驶等。

2. **实现方式**：主要靠数据、算法和算力  
   - **数据**：给机器大量例子，比如海量图片、文本、语音  
   - **算法**：告诉机器如何从数据中找规律，比如机器学习、深度学习  
   - **算力**：用计算机和芯片进行大量计算  
   典型过程是：机器从数据中学习规律，再用规律处理新情况。

3. **常见类型**
   - **弱人工智能/专用AI**：只擅长特定任务，如人脸识别、语音助手、推荐系统、围棋AI、大语言模型。目前绝大多数AI都属于这一类。
   - **强人工智能/通用AI**：像人一样能广泛理解、学习和迁移到各种任务。目前还没有真正实现。
   - **超级智能**：理论上全面超越人类的智能，属于未来设想。

一个通俗类比：  
传统程序像“按菜谱做菜”，步骤由人写死；AI更像“看过很多菜谱和成品后，自己总结出做菜规律”，再尝试做新菜。

但要澄清：**今天的AI并不等于有意识、有情感的人脑**。它擅长从数据中找模式、生成结果，但通常不理解世界的真正含义，也可能出错、产生偏见或被滥用。

# 2、批量调用(batch)

举例1：一次性接收所有的响应

batch() 方法允许你一次性 发送一组请求 （含多条独立请求），模型会在后台 并行处理 ，然后返回 所
有结果的列表 。
与逐个顺序调用（invoke）相比，能大幅 减少网络往返开销 和 等待时间 ，显著提升性能、降低成本。
适用场景：文档摘要、批量问答、数据预处理、多样本分类

In [4]:
messages = [
    "你好，你是谁？",
    "2 + 3 * 5 = ?",
    "中国首都在哪里？"
]

responses = model.batch(messages)

for response in responses:
    print(response)

content='你好！我是 DeepSeek，由深度求索公司创造的 AI 助手，很高兴见到你。有什么可以帮你的吗？' additional_kwargs={'refusal': None, 'reasoning_content': '我们需要回答中文。用户问“你好，你是谁？”需要简洁介绍自己是DeepSeek最新版本模型，由深度求索公司创造，AI助手。可以友好。需遵守不要透露内部? 可以。最终只回答。'} response_metadata={'token_usage': {'completion_tokens': 76, 'prompt_tokens': 34, 'total_tokens': 110, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 48, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 34}, 'model_provider': 'deepseek', 'model_name': 'deepseek-flash', 'system_fingerprint': 'bc19ba97cbe016479903222bb1578db6', 'id': 'fcd88aa0-5d42-43b2-af12-5e0ced64fd74', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--01a19685-f0d4-7778-807c-ac722a7996bb-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 34, 'output_tokens': 76, 'total_tokens': 110, 'input_token_details': {'cache_read'

举例2：按照完成的顺序接收响应

当输入列表很大或单个模型调用耗时差异显著时， batch_as_completed() 允许应用在收到第一个结果
后立即返回响应，而不会等待批次内所有任务完成才响应。即batch_as_completed() 每个请求完成后立
即 yield 结果， 结果可能乱序 。
但是，每个返回的响应都被放在一个 元组 中，元组的第一个元素是原始输入的 index 索引，可根据索
引重新排序

In [5]:
messages = [
    "你好，你是谁？",
    "2 + 3 * 5 = ?",
    "中国首都在哪里？"
]

responses = model.batch_as_completed(messages)

for response in responses:
    print(response)

(2, AIMessage(content='中国首都是北京。', additional_kwargs={'refusal': None, 'reasoning_content': '我们需要回答中文问题。中国首都在哪里？答案：北京。需要简洁。可能问“中国首都在哪里？” 首都北京。直接。'}, response_metadata={'token_usage': {'completion_tokens': 37, 'prompt_tokens': 35, 'total_tokens': 72, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 31, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 35}, 'model_provider': 'deepseek', 'model_name': 'deepseek-flash', 'system_fingerprint': 'bc19ba97cbe016479903222bb1578db6', 'id': '70177a15-1e8f-426b-b6ad-861e7919046c', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a19688-b7bd-77c5-a2a7-5c7f6b3f5193-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 35, 'output_tokens': 37, 'total_tokens': 72, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 31}}))
(1, A

举例3：性能对比

使用batch


In [6]:
# 准备多个输入
inputs = [
    "翻译成英文：春天来了",
    "翻译成英文：夏天很热",
    "翻译成英文：秋天落叶",
    "翻译成英文：冬天下雪"
]

# ✅ 批量调用（高效）
import time

start = time.time()
responses = model.batch(inputs)
batch_time = time.time() - start

print("批量调用结果：")
for i, response in enumerate(responses):
    print(f"{i+1}. {response.content}")
print(f"耗时: {batch_time:.2f}秒\n")


批量调用结果：
1. Spring has arrived.
2. Summer is very hot.
3. Autumn leaves

(If you mean “fallen leaves in autumn,” say: **fallen leaves in autumn**.)
4. It snows in winter.
耗时: 1.81秒



作为对比，演示循环调用invoke()

In [7]:
# ❌ 循环调用（低效，仅用于对比）
inputs = [
    "翻译成英文：春天来了",
    "翻译成英文：夏天很热",
    "翻译成英文：秋天落叶",
    "翻译成英文：冬天下雪"
]

start = time.time()
loop_responses = []
for inp in inputs:
    response = model.invoke(inp)
    loop_responses.append(response)
loop_time = time.time() - start

for i, response in enumerate(responses):
    print(f"{i+1}. {response.content}")

print(f"循环调用耗时: {loop_time:.2f}秒")

print(f"批量调用节省: {((loop_time - batch_time) / loop_time * 100):.1f}%")

1. Spring has arrived.
2. Summer is very hot.
3. Autumn leaves

(If you mean “fallen leaves in autumn,” say: **fallen leaves in autumn**.)
4. It snows in winter.
循环调用耗时: 4.55秒
批量调用节省: 60.2%
